In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# 1. DATASET LOADING

print("=" * 60)
print("1. DATASET LOADING")
print("=" * 60)

# Load dataset (handles both Google Colab and local Jupyter environments)
try:
    from google.colab import files
    print("Please upload your heart dataset CSV file:")
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
    heart_data = pd.read_csv(file_name)
except Exception:
    file_name = "heart.csv"
    print(f"Loading local file: {file_name}")
    heart_data = pd.read_csv(file_name)

# Clean column names (strip whitespace and convert to lowercase)
heart_data.columns = heart_data.columns.str.strip().str.lower()

target_column = 'target'

if target_column not in heart_data.columns:
    # Look for alternative target column names
    possible_targets = ['num', 'heartdisease', 'heart_disease', 'output', 'label', 'diagnosis']
    for col in possible_targets:
        if col in heart_data.columns:
            heart_data.rename(columns={col: target_column}, inplace=True)
            break

print(f"\nDataset loaded successfully from: {file_name}")
print(f"Dataset Shape : {heart_data.shape[0]} rows, {heart_data.shape[1]} columns")
print("Target Column :", target_column)

print("\nFirst 5 Rows of Dataset:")
display(heart_data.head())


# 2. DATA PREPROCESSING

print("\n" + "=" * 60)
print("2. DATA PREPROCESSING")
print("=" * 60)

# Check categorical and numerical feature representation
print("Checking feature representation...")
print("Numerical representation confirmed: Categorical/binary features are already encoded numerically.")

# Check for duplicate rows
duplicate_count = heart_data.duplicated().sum()
if duplicate_count > 0:
    print(f"Found {duplicate_count} duplicate row(s). Removing duplicates...")
    heart_data = heart_data.drop_duplicates().reset_index(drop=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

# Remove rows missing the target label
heart_data = heart_data.dropna(subset=[target_column]).copy()

# Separate features (X) and target (y)
X = heart_data.drop(columns=[target_column])
y = heart_data[target_column]

# Convert features and target to numeric values where necessary
X = X.apply(pd.to_numeric, errors='coerce')
y = pd.to_numeric(y, errors='coerce')

# Ensure binary target classification (0 = Healthy, 1 = Heart Disease)
y = (y > 0).astype(int)

# Check missing values in features
missing_count = X.isnull().sum().sum()
if missing_count > 0:
    print(f"Found {missing_count} missing value(s) in feature matrix.")
    print("Missing values per column:")
    print(X.isnull().sum()[X.isnull().sum() > 0])
else:
    print("No missing values found.")

print("\nTarget Value Counts:")
print(y.value_counts().rename({0: '0 (Healthy)', 1: '1 (Heart Disease)'}))


# 3. FEATURE SCALING

print("\n" + "=" * 60)
print("3. FEATURE SCALING")
print("=" * 60)

print("StandardScaler object initialized.")
print("Note: Scaling will be fitted ONLY on the training split in Step 4 to prevent data leakage.")


# 4. DATASET SPLITTING


print("\n" + "=" * 60)
print("4. DATASET SPLITTING")
print("=" * 60)

# Split dataset into 80% Training and 20% Testing (Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=2
)

print(f"Full Feature Matrix Shape : {X.shape}")
print(f"Training Set Shape (80%)  : {X_train.shape}")
print(f"Testing Set Shape (20%)   : {X_test.shape}")

# Fit Imputer strictly on training data
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

# Fit StandardScaler strictly on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nImputation and Feature Scaling successfully applied strictly after dataset splitting.")


# 5. SVM MODEL TRAINING

print("\n" + "=" * 60)
print("5. SVM MODEL TRAINING")
print("=" * 60)

kernels = ['linear', 'poly', 'rbf']
models = {}

for kernel in kernels:
    print(f"Training Support Vector Classifier with '{kernel}' kernel...")
    model = SVC(kernel=kernel)
    model.fit(X_train_scaled, y_train)
    models[kernel] = model

print("All SVM models trained successfully.")


# 6. KERNEL COMPARISON

print("\n" + "=" * 60)
print("6. KERNEL COMPARISON")
print("=" * 60)

results = []

for kernel in kernels:
    model = models[kernel]

    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    precision = precision_score(y_test, test_pred, zero_division=0)
    recall = recall_score(y_test, test_pred, zero_division=0)
    f1 = f1_score(y_test, test_pred, zero_division=0)

    results.append({
        'Kernel': kernel.capitalize(),
        'Training Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

results_df = pd.DataFrame(results)
display(results_df)

# Select final model based strictly on maximum Test Accuracy
best_idx = results_df['Test Accuracy'].idxmax()
best_kernel_name = results_df.loc[best_idx, 'Kernel'].lower()
final_model = models[best_kernel_name]

print(f"\nSelected Optimal Model: SVM with '{best_kernel_name.upper()}' Kernel (Highest Test Accuracy)")


# 7. FINAL MODEL EVALUATION

print("\n" + "=" * 60)
print("7. FINAL MODEL EVALUATION")
print("=" * 60)

final_predictions = final_model.predict(X_test_scaled)

final_accuracy = accuracy_score(y_test, final_predictions)
final_precision = precision_score(y_test, final_predictions, zero_division=0)
final_recall = recall_score(y_test, final_predictions, zero_division=0)
final_f1 = f1_score(y_test, final_predictions, zero_division=0)

print(f"Final Model Kernel : {best_kernel_name.upper()}")
print(f"Accuracy           : {final_accuracy:.4f} ({final_accuracy * 100:.2f}%)")
print(f"Precision          : {final_precision:.4f}")
print(f"Recall             : {final_recall:.4f}")
print(f"F1-Score           : {final_f1:.4f}")


# 8. CONFUSION MATRIX

print("\n" + "=" * 60)
print("8. CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(y_test, final_predictions)

tn, fp, fn, tp = cm.ravel()

print(f"True Negative (Healthy correctly predicted)           : {tn}")
print(f"False Positive (Healthy wrongly predicted as Disease) : {fp}")
print(f"False Negative (Disease wrongly predicted as Healthy) : {fn}")
print(f"True Positive (Disease correctly predicted)           : {tp}")

# Plotting Confusion Matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Healthy', 'Heart Disease']
)
disp.plot(cmap=plt.cm.Blues)
plt.title(f'Confusion Matrix - {best_kernel_name.upper()} SVM')
plt.show()


# 9. FINAL HEART DISEASE PREDICTION

print("\n" + "=" * 60)
print("9. FINAL HEART DISEASE PREDICTION")
print("=" * 60)

# Sample patient feature vector (13 clinical features)
# Features: [age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal]
sample_patient = (58, 0, 0, 170, 225, 1, 0, 146, 1, 2.8, 1, 2, 1)

print("Sample Patient Features:")
print(sample_patient)

# Reshape into a 2D sample array
patient_array = np.asarray(sample_patient, dtype=float).reshape(1, -1)

if patient_array.shape[1] != X.shape[1]:
    print(f"\nERROR: Provided input has {patient_array.shape[1]} features, but dataset expects {X.shape[1]} features.")
else:
    # Scale patient data using the fitted StandardScaler
    patient_scaled = scaler.transform(patient_array)

    # Predict label
    patient_pred = final_model.predict(patient_scaled)[0]

    print(f"\nPredicted Class Output: {patient_pred}")
    if patient_pred == 0:
        print("Prediction Result     : The patient is predicted as HEALTHY (No Heart Disease detected).")
    else:
        print("Prediction Result     : The patient is predicted as having HEART DISEASE.")


# 10. PROJECT SUMMARY

#print("\n" + "=" * 60)
print("10. PROJECT SUMMARY")
#print("=" * 60)

print("Project Name        : Heart Disease Prediction Using Support Vector Machine (SVM)")
print("Algorithm Used      : Support Vector Classifier (SVC)")
print(f"Optimal Kernel      : {best_kernel_name.upper()}")
print(f"Test Accuracy       : {final_accuracy:.4f} ({final_accuracy * 100:.2f}%)")
print(f"Precision           : {final_precision:.4f}")
print(f"Recall              : {final_recall:.4f}")
print(f"F1-Score            : {final_f1:.4f}")
print("Data Leakage Check  : Passed (Imputer and Scaler fitted solely on X_train)")
print("Workflow Execution  : Completed Successfully")